In [1]:
print("AniLens ML is running!")


AniLens ML is running!


In [ ]:
###########################################################data preprocessing####################################################

In [11]:
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 35)
pd.set_option('display.max_colwidth', 180)

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5.5)
plt.rcParams['font.size'] = 13

print("Environment ready – AniLens TF-IDF recommender notebook")

Environment ready – AniLens TF-IDF recommender notebook


In [2]:
DATA_FILE = "dataset/anime.csv"

df = pd.read_csv(DATA_FILE, low_memory=False)

print("Dataset shape:      ", df.shape)
print("Number of rows:     ", len(df))
print("Number of columns:  ", df.shape[1])
print("\nColumns:\n", df.columns.tolist())

df.head(3)

Dataset shape:       (28858, 58)
Number of rows:      28858
Number of columns:   58

Columns:
 ['mal_id', 'url', 'approved', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'image_jpg_url', 'image_jpg_small_url', 'image_jpg_large_url', 'image_webp_url', 'image_webp_small_url', 'image_webp_large_url', 'trailer_youtube_id', 'trailer_url', 'trailer_embed_url', 'trailer_image_url', 'trailer_small_image_url', 'trailer_medium_image_url', 'trailer_large_image_url', 'trailer_maximum_image_url', 'type', 'source', 'episodes', 'status', 'airing', 'duration', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites', 'synopsis', 'background', 'aired_from', 'aired_to', 'aired_prop_from_day', 'aired_prop_from_month', 'aired_prop_from_year', 'aired_prop_to_day', 'aired_prop_to_month', 'aired_prop_to_year', 'aired_string', 'season', 'year', 'broadcast_day', 'broadcast_time', 'broadcast_timezone', 'broadcast_string', 'producers', 'licensors', 'studios', 'genres', 'expli

,mal_id,url,approved,title,title_english,title_japanese,title_synonyms,image_jpg_url,image_jpg_small_url,image_jpg_large_url,...,broadcast_time,broadcast_timezone,broadcast_string,producers,licensors,studios,genres,explicit_genres,themes,demographics
0,1,https://myanimelist.net/anime/1/Cowboy_Bebop,True,Cowboy Bebop,Cowboy Bebop,カウボーイビバップ,NaN,https://cdn.myanimelist.net/images/anime/4/196...,https://cdn.myanimelist.net/images/anime/4/196...,https://cdn.myanimelist.net/images/anime/4/196...,...,01:00,Asia/Tokyo,Saturdays at 01:00 (JST),"Bandai Visual, Victor Entertainment, Audio Pla...",Funimation,Sunrise,"Action, Award Winning, Sci-Fi",NaN,"Adult Cast, Space",NaN
1,5,https://myanimelist.net/anime/5/Cowboy_Bebop__...,True,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,カウボーイビバップ 天国の扉,Cowboy Bebop: Knockin' on Heaven's Door,https://cdn.myanimelist.net/images/anime/1439/...,https://cdn.myanimelist.net/images/anime/1439/...,https://cdn.myanimelist.net/images/anime/1439/...,...,NaN,NaN,NaN,"Sunrise, Bandai Visual","Sony Pictures Entertainment, Funimation",Bones,"Action, Sci-Fi",NaN,"Adult Cast, Space",NaN
2,6,https://myanimelist.net/anime/6/Trigun,True,Trigun,Trigun,トライガン,NaN,https://cdn.myanimelist.net/images/anime/1130/...,https://cdn.myanimelist.net/images/anime/1130/...,https://cdn.myanimelist.net/images/anime/1130/...,...,01:15,Asia/Tokyo,Thursdays at 01:15 (JST),Victor Entertainment,Funimation,Madhouse,"Action, Adventure, Sci-Fi",NaN,Adult Cast,Shounen


In [3]:
print("Missing values per column:")
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

print("\nAnime types distribution:")
print(df['type'].value_counts(dropna=False))

Missing values per column:
explicit_genres              28858
background                   25957
broadcast_day                24696
broadcast_time               24696
broadcast_timezone           24696
licensors                    23721
trailer_url                  23305
trailer_medium_image_url     23305
trailer_small_image_url      23305
trailer_youtube_id           23305
trailer_image_url            23305
trailer_large_image_url      23305
trailer_maximum_image_url    23305
trailer_embed_url            23304
year                         22638
season                       22638
broadcast_string             20459
demographics                 18093
aired_to                     17743
aired_prop_to_day            17743
aired_prop_to_year           17743
aired_prop_to_month          17743
title_english                16423
producers                    15236
title_synonyms               14592
themes                       11838
studios                      11671
scored_by                   

In [4]:
# Keep only columns useful for recommendation
useful_columns = [
    'mal_id', 'title', 'title_english', 'synopsis', 'genres',
    'themes', 'studios', 'image_url', 'score', 'scored_by',
    'type', 'episodes', 'status', 'source'
]

available = [c for c in useful_columns if c in df.columns]
df = df[available].copy()

print("Columns kept:", df.columns.tolist())

# Drop rows missing title or synopsis
df.dropna(subset=['title', 'synopsis'], inplace=True)

# Keep only the most common watchable formats
watchable_types = ['TV', 'Movie', 'OVA', 'ONA', 'Special']
df = df[df['type'].isin(watchable_types)].copy()

print("\nAfter filtering:")
print("→ Shape:", df.shape)
print("→ Types remaining:", df['type'].value_counts().index.tolist())

Columns kept: ['mal_id', 'title', 'title_english', 'synopsis', 'genres', 'themes', 'studios', 'score', 'scored_by', 'type', 'episodes', 'status', 'source']

After filtering:
→ Shape: (18410, 13)
→ Types remaining: ['TV', 'OVA', 'Movie', 'ONA', 'Special']


In [5]:
# Clean synopsis
def clean_synopsis(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)          # remove HTML tags
    text = re.sub(r'\[.*?\]', '', text)          # remove [Written by ...]
    text = re.sub(r'[^a-z0-9\s\'-]', '', text)   # keep letters, numbers, space, apostrophe, hyphen
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['synopsis_clean'] = df['synopsis'].apply(clean_synopsis)

# Show example before / after
pd.DataFrame({
    'Original': df['synopsis'].head(3),
    'Cleaned': df['synopsis_clean'].head(3)
})

,Original,Cleaned
0,"Crime is timeless. By the year 2071, humanity ...",crime is timeless by the year 2071 humanity ha...
1,"Another day, another bounty—such is the life o...",another day another bountysuch is the life of ...
2,"Vash the Stampede is the man with a $$60,000,0...",vash the stampede is the man with a 6000000000...


In [6]:
# ─── Helper to parse comma-separated or list-like columns ──────────────────────

def parse_comma_or_list(value, sep=','):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x).strip().replace(" ", "") for x in value]
    if isinstance(value, str):
        return [x.strip().replace(" ", "") for x in value.split(sep) if x.strip()]
    return []

# Apply parsing
df['genres_clean']  = df['genres'].apply(parse_comma_or_list)
df['themes_clean']  = df['themes'].apply(parse_comma_or_list)
df['studios_clean'] = df['studios'].apply(parse_comma_or_list)

# ─── Create weighted combined text representation ──────────────────────────────

df['anime_dna'] = (
    df['synopsis_clean'].str.split() +             # natural language
    (df['genres_clean']  * 3) +                    # genres are very strong signal
    df['themes_clean'] +                           # psychological, isekai, mecha...
    df['studios_clean']                            # animation studio style
).apply(lambda x: " ".join(x))

print("Example anime DNA (first 3):")
for i in range(3):
    print(f"→ {df.iloc[i]['title']}")
    print("  ", df.iloc[i]['anime_dna'][:180] + "..." if len(df.iloc[i]['anime_dna']) > 180 else df.iloc[i]['anime_dna'])
    print()

Example anime DNA (first 3):
→ Cowboy Bebop
   crime is timeless by the year 2071 humanity has expanded across the galaxy filling the surface of other planets with settlements like those on earth these new societies are plagued...

→ Cowboy Bebop: Tengoku no Tobira
   another day another bountysuch is the life of the often unlucky crew of the bebop however this routine is interrupted when faye who is chasing a fairly worthless target on mars wit...

→ Trigun
   vash the stampede is the man with a 60000000000 bounty on his head the reason he's a merciless villain who lays waste to all those that oppose him and flattens entire cities for fu...



In [12]:
# ─── TF-IDF Vectorizer ─────────────────────────────────────────────────────────

vectorizer = TfidfVectorizer(
    max_features       = 7000,
    stop_words         = 'english',
    ngram_range        = (1, 2),
    min_df             = 4,
    max_df             = 0.90,
    norm               = 'l2',
    smooth_idf         = True
)

print("Fitting TF-IDF vectorizer...")
tfidf_matrix = vectorizer.fit_transform(df['anime_dna'])

print("→ TF-IDF matrix shape:", tfidf_matrix.shape)
print("→ Vocabulary size:", len(vectorizer.get_feature_names_out()))

# ─── Cosine similarity matrix ─────────────────────────────────────────────────

print("\nComputing cosine similarity...")
similarity = cosine_similarity(tfidf_matrix)

print("→ Similarity matrix shape:", similarity.shape)

Fitting TF-IDF vectorizer...
→ TF-IDF matrix shape: (18410, 7000)
→ Vocabulary size: 7000

Computing cosine similarity...
→ Similarity matrix shape: (18410, 18410)


In [8]:
def recommend_anime(
    title,
    n_recommendations=7,
    show_poster_url=True,
    min_similarity=0.08
):
    """
    Recommend similar anime based on TF-IDF + cosine similarity.
    """
    if title not in df['title'].values:
        print(f"Anime '{title}' not found in dataset.")
        return []

    idx = df[df['title'] == title].index[0]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    print(f"\n{'═'*70}")
    print(f"  Recommendations for:  {title}")
    print(f"{'═'*70}\n")

    results = []
    count = 0

    for rank, (i, sim) in enumerate(sim_scores[1:], 1):
        if sim < min_similarity:
            break
        if count >= n_recommendations:
            break

        row = df.iloc[i]
        print(f"#{rank:2d}  {row['title']}")
        print(f"     Similarity: {sim:.4f}")
        print(f"     Genres:     {', '.join(row['genres_clean'][:6])}")
        if show_poster_url and 'image_url' in row:
            print(f"     Poster:     {row['image_url']}")
        print()

        results.append({
            'rank': rank,
            'title': row['title'],
            'similarity': round(sim, 4),
            'image_url': row.get('image_url', ''),
            'genres': row['genres_clean'],
            'type': row.get('type', ''),
            'score': row.get('score', np.nan)
        })
        count += 1

    return results

In [13]:
# ─── Demonstration for presentation ────────────────────────────────────────────

print("Example recommendations:\n")

recommend_anime("Sousou no Frieren", n_recommendations=8)
recommend_anime("Shingeki no Kyojin", n_recommendations=6)
recommend_anime("Death Note")
recommend_anime("Jujutsu Kaisen")
recommend_anime("One Piece")

Example recommendations:


══════════════════════════════════════════════════════════════════════
  Recommendations for:  Sousou no Frieren
══════════════════════════════════════════════════════════════════════

# 1  Guomin Laogong Dai Huijia 3rd Season
     Similarity: 0.5972
     Genres:     Drama, Romance

# 2  Kizuna (Special)
     Similarity: 0.5966
     Genres:     Drama, Romance

# 3  Tooi Sekai
     Similarity: 0.5820
     Genres:     Drama, Romance

# 4  Hudie Meng: Liang Shan Bo yu Zhu Ying Tai
     Similarity: 0.5676
     Genres:     Drama, Romance

# 5  Douse, Koishite Shimaunda. Season 2
     Similarity: 0.5523
     Genres:     Drama, Romance

# 6  Douse, Koishite Shimaunda. Season 2
     Similarity: 0.5523
     Genres:     Drama, Romance

# 7  Candy Candy: Candy no Natsu Yasumi
     Similarity: 0.5497
     Genres:     Drama, Romance

# 8  Accept
     Similarity: 0.5448
     Genres:     Drama, Romance


══════════════════════════════════════════════════════════════════════

[{'rank': 1,
  'title': 'One Piece: Gyojin Tou-hen',
  'similarity': np.float64(0.4747),
  'image_url': '',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'type': 'TV',
  'score': np.float64(7.92)},
 {'rank': 2,
  'title': 'One Piece: Gyojin Tou-hen',
  'similarity': np.float64(0.4747),
  'image_url': '',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'type': 'TV',
  'score': np.float64(7.92)},
 {'rank': 3,
  'title': 'One Piece: Romance Dawn Story',
  'similarity': np.float64(0.4509),
  'image_url': '',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'type': 'OVA',
  'score': np.float64(7.33)},
 {'rank': 4,
  'title': 'One Piece Movie 02: Nejimaki-jima no Daibouken',
  'similarity': np.float64(0.4213),
  'image_url': '',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'type': 'Movie',
  'score': np.float64(7.08)},
 {'rank': 5,
  'title': 'One Piece: Taose! Kaizoku Ganzack',
  'similarity': np.float64(0.3921),
  'image_url': '',
  'genres': ['Action', 'Adventure', 'Fantasy'],

In [16]:
# ─── FINAL STEP: Save Artifacts – Robust & Safe Version ────────────────────────

print("\nPreparing final data for Flask...")

# 1. Detect and rename image column if present
image_col_candidates = [
    'image_url', 
    'main_picture', 
    'poster', 
    'images.jpg.image_url', 
    'cover_image', 
    'picture_url',
    'main_picture.medium',
    'main_picture.large'
]

image_col_found = None
for cand in image_col_candidates:
    if cand in df.columns:
        image_col_found = cand
        break

if image_col_found and image_col_found != 'image_url':
    print(f"→ Found image column '{image_col_found}' → renaming to 'image_url'")
    df = df.rename(columns={image_col_found: 'image_url'})
elif image_col_found:
    print("→ Image column 'image_url' already exists")
else:
    print("→ Warning: No image column detected → Flask will use placeholders")

# 2. Detect and rename genres column if present
genre_col_candidates = [
    'genres', 
    'genres_clean', 
    'genre', 
    'genres_list', 
    'genre_list'
]

genre_col_found = None
for cand in genre_col_candidates:
    if cand in df.columns:
        genre_col_found = cand
        break

if genre_col_found and genre_col_found != 'genres':
    print(f"→ Found genres column '{genre_col_found}' → renaming to 'genres'")
    df = df.rename(columns={genre_col_found: 'genres'})
elif genre_col_found:
    print("→ Genres column 'genres' already exists")
else:
    print("→ Warning: No genres column detected → recommendations will show '—' for genres")

# 3. Build final_data with only existing columns
core_cols = ['mal_id', 'title', 'type', 'score']

if 'image_url' in df.columns:
    core_cols.append('image_url')

if 'genres' in df.columns:
    core_cols.append('genres')

# Only keep columns that actually exist
final_cols = [c for c in core_cols if c in df.columns]

if not final_cols:
    raise ValueError("No usable columns found! Check your DataFrame.")

final_data = df[final_cols].copy()

# 4. Show what we're saving (very useful for debugging / presentation)
print("\n" + "═"*60)
print("FINAL DATAFRAME TO BE SAVED")
print("Columns:", final_data.columns.tolist())
print("Shape:", final_data.shape)
print("First row preview:")
display(final_data.head(1))
print("═"*60 + "\n")

# 5. Save the artifacts
final_data.to_pickle("anilens_data.pkl")
pickle.dump(similarity, open("anilens_similarity.pkl", "wb"))
pickle.dump(vectorizer, open("anilens_vectorizer.pkl", "wb"))

print("""
╔════════════════════════════════════════════════════════════╗
║                 ARTIFACTS SAVED SUCCESSFULLY               ║
╠════════════════════════════════════════════════════════════╣
║  • anilens_data.pkl          (anime metadata)              ║
║  • anilens_similarity.pkl    (precomputed similarity)      ║
║  • anilens_vectorizer.pkl    (fitted TF-IDF vectorizer)    ║
╚════════════════════════════════════════════════════════════╝

Next: You can now run your Flask app!
""")


Preparing final data for Flask...
→ Warning: No image column detected → Flask will use placeholders
→ Genres column 'genres' already exists

════════════════════════════════════════════════════════════
FINAL DATAFRAME TO BE SAVED
Columns: ['mal_id', 'title', 'type', 'score', 'genres']
Shape: (18410, 5)
First row preview:


,mal_id,title,type,score,genres
0,1,Cowboy Bebop,TV,8.75,"Action, Award Winning, Sci-Fi"


════════════════════════════════════════════════════════════


╔════════════════════════════════════════════════════════════╗
║                 ARTIFACTS SAVED SUCCESSFULLY               ║
╠════════════════════════════════════════════════════════════╣
║  • anilens_data.pkl          (anime metadata)              ║
║  • anilens_similarity.pkl    (precomputed similarity)      ║
║  • anilens_vectorizer.pkl    (fitted TF-IDF vectorizer)    ║
╚════════════════════════════════════════════════════════════╝

Next: You can now run your Flask app!

